In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import dm4bem

In [2]:
# Rooms geometry
l = 3               # m length of the cubic room
L = 5
w = 0.2
H = 3

# Window geometry
l_window = 1.25
L_window = 2
w_window = 0.07

# Doors geometry
l_door = 2.00
L_door = 0.90
w_door = 0.1

# Surfaces
S_window = l_window * L_window
S_door = l_door * L_door
S_wall_total = H * (2*L + 2*w + l)            # m² surface area of wall
S_wall_1 = S_wall_total - S_door
S_wall_2 = S_wall_total - S_door - S_window
S_wall_3 = l*H

# Temperatures

To = -5     # degC
T1_sp = 25  # degC
E = 200     # W/m²

In [3]:
air = {'Density': 1.2,                      # kg/m³
       'Specific heat': 1000}               # J/(kg·K)
pd.DataFrame(air, index=['Air'])

,Density,Specific heat
Air,1.2,1000


In [4]:
concrete = {'Conductivity': 1.740,          # W/(m·K)
            'Density': 2300.0,              # kg/m³
            'Specific heat': 880,           # J/(kg⋅K)
            'Width': w*0.8}                   # m

insulation = {'Conductivity': 0.035,        # W/(m·K)
              'Density': 55.0,              # kg/m³
              'Specific heat': 1210,        # J/(kg⋅K)
              'Width': w*0.2}                 # m

double_glass = {'Conductivity': 0.112,               # W/(m·K)
         'Density': 2500,                   # kg/m³
         'Specific heat': 1210,             # J/(kg⋅K)
         'Width': w_window,                 #m
         'Surface': S_window}               #m²

solid_wood = {'Conductivity': 0.14,               # W/(m·K)
         'Density': 2500,                   # kg/m³
         'Specific heat': 1210,             # J/(kg⋅K)
         'Width': w_door,                   # m
         'Surface': S_door}                 # m²

surfaces = {
    'Layer_out_1': S_wall_1,
    'Layer_in_1': S_wall_1,
    'Layer_out_2': S_wall_2,
    'Layer_in_2': S_wall_2,
    'Layer_out_3': S_wall_3,
    'Layer_in_3': S_wall_3,
}
 
wall = pd.DataFrame.from_dict({'Layer_out_1': concrete,
                               'Layer_in_1': insulation,
                               'Layer_out_2': concrete,
                               'Layer_in_2': insulation,
                               'Layer_out_3': concrete,
                               'Layer_in_3': insulation,
                               'Window': double_glass,
                               'Door': solid_wood},
                              orient='index')

wall['Surface'] = wall['Surface'].fillna(pd.Series(surfaces))

wall

,Conductivity,Density,Specific heat,Width,Surface
Layer_out_1,1.740,2300.0,880,0.16,38.4
Layer_in_1,0.035,55.0,1210,0.04,38.4
Layer_out_2,1.740,2300.0,880,0.16,35.9
Layer_in_2,0.035,55.0,1210,0.04,35.9
Layer_out_3,1.740,2300.0,880,0.16,9.0
Layer_in_3,0.035,55.0,1210,0.04,9.0
Window,0.112,2500.0,1210,0.07,2.5
Door,0.140,2500.0,1210,0.10,1.8


In [5]:
h = pd.DataFrame([{'in': 8., 'out': 25}], index=['h'])  # W/(m²⋅K)
h

,in,out
h,8.0,25


### Conduction conductances

$$G_{cd} = \frac{\lambda}{w}S$$

where:

- $\lambda$ - [thermal conductvity](https://en.m.wikipedia.org/wiki/Thermal_conductivity), W/(m⋅K);
- $w$ - width of the material, m;
- $S$ - surface area of the wall, m².

In [6]:
G_cd = wall['Conductivity'] / wall['Width'] * wall['Surface']
pd.DataFrame(G_cd, columns=['Conductance'])

,Conductance
Layer_out_1,417.6000
Layer_in_1,33.6000
Layer_out_2,390.4125
Layer_in_2,31.4125
Layer_out_3,97.8750
Layer_in_3,7.8750
Window,4.0000
Door,2.5200


### Convection conductances
$$G_{cv} = {h S}$$

where:
- $h$ is the [convection coefficient](https://en.m.wikipedia.org/wiki/Heat_transfer_coefficient), W/(m²⋅K);
- $S$ - surface area of the wall, m².

In [7]:
G_wall_1 = h * wall['Surface'].iloc[0]
G_wall_2 = h * wall['Surface'].iloc[2] 
G_wall_3 = h * wall['Surface'].iloc[4] 
G_window = h * wall['Surface'].iloc[6]     
G_door = h * wall['Surface'].iloc[7]

### Advection conductances

$$\dot{V}_a = \frac{\mathrm{ACH}}{3600} V_a$$

where:
- $\mathrm{ACH}$  ([air changes per hour](https://en.m.wikipedia.org/wiki/Air_changes_per_hour)) is the air infiltration rate, 1/h;
- $3600$ - number of seconds in one hour, s/h;
- $V_a$ - volume of the air in the thermal zone, m³.

Therefore, the conductance of [advection](https://en.m.wikipedia.org/wiki/Advection) by [ventilation](https://en.m.wikipedia.org/wiki/Ventilation_(architecture)) and/or [infiltration](https://en.m.wikipedia.org/wiki/Infiltration_(HVAC)), in W/K, is:

$$G_v = \rho_a c_a \dot{V}_a$$

In [8]:
Va = (L*l*H)                   # m³, volume of air
ACH = 1                     # 1/h, air changes per hour
Va_dot = ACH / 3600 * Va    # m³/s, air infiltration

Gv = air['Density'] * air['Specific heat'] * Va_dot

### Proportional controller

$$ q_{HVAC} = K_p (T_{i, sp} - \theta_i)$$

where:
- $K_p$ is the proportional gain of the controller, W/K;
- $T_{i, sp}$ - indoor temperature [setpoint](https://en.m.wikipedia.org/wiki/Setpoint_(control_system)), °C (noted in majuscule because it is an *input*, i.e., independent, variable);
- $\theta_i$ - indoor temperature, °C (noted in minuscule because it is an *output*, i.e., dependent variable).

In [9]:
# P-controler gain
# Kp = 1e4            # almost perfect controller Kp -> ∞
# Kp = 1e-3           # no controller Kp -> 0
Kp = 0

### Conductances in series and/or parallel

$$ G_{gs} = \frac{1}{1/G_{g,cv.out } + 1/(2 G_{g,cd})} =  
\frac{1}{\frac{1}{h_{out} S_g} + \frac{w / 2}{\lambda S_g}}
$$

In [10]:
Ggs_window = float(1 / (1 / G_window.loc['h', 'out'] + 1 / (2 * G_cd['Window'])))
Ggs_door = float(1 / (1 / G_window.loc['h', 'out'] + 1 / (2 * G_cd['Door'])))

Ggs_1 = Ggs_door
Ggs_2 = Ggs_door + Ggs_window

#### Thermal capacities

$$C_w= m_w c_w= \rho_w c_w w_w S_w$$

where:
- $m_w = \rho_w w_w S_w$ is the mass of the wall, kg;
- $c_w$ - [specific heat capacity](https://en.m.wikipedia.org/wiki/Specific_heat_capacity), J/(kg⋅K);
- $\rho_w$ - [density](https://en.m.wikipedia.org/wiki/Density), kg/m³;
- $w_w$ - width of the wall, m;
- $S_w$ - surface area of the wall, m².

In [11]:
C = wall['Density'] * wall['Specific heat'] * wall['Surface'] * wall['Width']
pd.DataFrame(C, columns=['Capacity'])

C['Air'] = air['Density'] * air['Specific heat'] * Va
pd.DataFrame(C, columns=['Capacity'])

,Capacity
Layer_out_1,12435456.0
Layer_in_1,102220.8
Layer_out_2,11625856.0
Layer_in_2,95565.8
Layer_out_3,2914560.0
Layer_in_3,23958.0
Window,529375.0
Door,544500.0
Air,54000.0


#### Thermal circuit

In [12]:
# temperature nodes
nθ = 8      # number of temperature nodes
θ = [f'θ{i}' for i in range(nθ)]
print(θ)

# flow-rate branches
nq = 13     # number of flow branches
q = [f'q{i}' for i in range(nq)]
print(q)

['θ0', 'θ1', 'θ2', 'θ3', 'θ4', 'θ5', 'θ6', 'θ7']
['q0', 'q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 'q12']


In [13]:
neglect_air_glass = False

if neglect_air_glass:
    C = np.array([0, 0, C['Layer_out_1'], C['Layer_in_1'], C['Layer_out_3'], C['Layer_in_3'],
                  C['Layer_out_2'], C['Layer_in_2']])
else:
    C = np.array([C['Air'], C['Air'], C['Layer_out_1'], C['Layer_in_1'], C['Layer_out_3'], C['Layer_in_3'],
                  C['Layer_out_2'], C['Layer_in_2']])

pd.DataFrame(C, index=θ)

,0
θ0,54000.0
θ1,54000.0
θ2,12435456.0
θ3,102220.8
θ4,2914560.0
θ5,23958.0
θ6,11625856.0
θ7,95565.8


#### A: incidence matrix

In [14]:
A = np.zeros([nq, nθ])       # n° of branches X n° of nodes

A[0, 2] = 1
A[1, 7] = 1
A[2, 0] = 1
A[3, 1] = 1
A[4, 0], A[4, 3] = -1, 1
A[5, 0], A[5, 4] = -1, 1
A[6, 5], A[6, 1] = -1, 1
A[7, 1], A[7, 6] = -1, 1
A[8, 2], A[8, 3] = -1, 1
A[9, 4], A[9, 5] = -1, 1
A[10, 7], A[10, 6] = -1, 1
A[11, 1] = 1
A[12, 0] = 1

pd.DataFrame(A, index=q, columns=θ)

,θ0,θ1,θ2,θ3,θ4,θ5,θ6,θ7
q0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
q1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
q2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
q3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
q4,-1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
q5,-1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
q6,0.0,1.0,0.0,0.0,0.0,-1.0,0.0,0.0
q7,0.0,-1.0,0.0,0.0,0.0,0.0,1.0,0.0
q8,0.0,0.0,-1.0,1.0,0.0,0.0,0.0,0.0
q9,0.0,0.0,0.0,0.0,-1.0,1.0,0.0,0.0


#### G: conductance matrix

In [15]:
G = np.array(np.hstack(
    [G_wall_1['out'],
     G_wall_2['out'],
     Ggs_1, #G_door['out'],
     Ggs_2, #G_door['out'] + G_window['out'],
     G_wall_1['in'],
     G_wall_3['in'],
     G_wall_3['in'],
     G_wall_2['in'],
     G_cd['Layer_in_1'] + G_cd['Layer_out_1'],
     G_cd['Layer_in_3'] + G_cd['Layer_out_3'],
     G_cd['Layer_in_2'] + G_cd['Layer_out_2'],
     Gv,
     Kp]))

pd.DataFrame(G, index=q)

,0
q0,960.000000
q1,897.500000
q2,4.663903
q3,11.756101
q4,307.200000
q5,72.000000
q6,72.000000
q7,287.200000
q8,451.200000
q9,105.750000


#### b: temperature source vector

In [16]:
b = pd.Series([To, To, To, To, 0, 0, 0, 0, 0, 0, 0, To, T1_sp],
              index=q)

pd.DataFrame(b, index=q)

,0
q0,-5
q1,-5
q2,-5
q3,-5
q4,0
q5,0
q6,0
q7,0
q8,0
q9,0


#### f: heat flow source vector

In [17]:
f = pd.Series([0, 0, E*S_wall_total, E*S_wall_total, E*S_wall_3, E*S_wall_3, E*S_wall_total, E*S_wall_total],
              index=θ)
pd.DataFrame(f, index=θ)

,0
θ0,0.0
θ1,0.0
θ2,8040.0
θ3,8040.0
θ4,1800.0
θ5,1800.0
θ6,8040.0
θ7,8040.0


#### y: output vector

In [18]:
y = np.zeros(nθ)         # nodes
y[[0]] = y [[1]] = 1              # nodes (temperatures) of interest
pd.DataFrame(y, index=θ)

,0
θ0,1.0
θ1,1.0
θ2,0.0
θ3,0.0
θ4,0.0
θ5,0.0
θ6,0.0
θ7,0.0
